In [ ]:
# Setup
from pathlib import Path
import sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_data
from model import (temporal_split, add_date_features, add_lag_features,
                   train_lightgbm, lightgbm_forecast, evaluate_forecast,
                   load_model, FEATURE_COLS_LGB)
from utils import save_fig, plot_forecast_vs_actual, plot_model_comparison

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
with open(ARTEFACTS_DIR / 'forecast_results.json') as f:
    results = json.load(f)
print("Models evaluated:", list(results.keys()))

In [ ]:
# 1. Reload LightGBM model + test split
df = load_data()
df = add_date_features(df)
df = add_lag_features(df)
train, test = temporal_split(df, test_days=56)

try:
    model = load_model('lightgbm_model', ARTEFACTS_DIR)
except Exception:
    print("Training model from scratch...")
    model = train_lightgbm(train)

lgbm_pred = lightgbm_forecast(model, train, test)
lgbm_metrics = evaluate_forecast(test['Sales'], lgbm_pred)
print(f"LightGBM — RMSPE = {lgbm_metrics['rmspe']:.2f}%  MAE = {lgbm_metrics['mae']:.0f}")

In [ ]:
# 2. Forecast vs actual for example store
store_to_plot = test['Store'].iloc[0]
plot_forecast_vs_actual(test, lgbm_pred, store=store_to_plot)
save_fig('04_forecast_vs_actual.png')
plt.show()

In [ ]:
# 3. Model comparison plot
axes = plot_model_comparison(results)
save_fig('05_model_comparison.png')
plt.show()

print("\nFull comparison table:")
print(pd.DataFrame(results).T[['rmspe', 'mape', 'mae', 'rmse']].round(2))

In [ ]:
# 4. LightGBM feature importance
importance = pd.DataFrame({
    'feature': FEATURE_COLS_LGB,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=importance.head(15), y='feature', x='importance', palette='viridis', ax=ax)
ax.set_title('LightGBM — top 15 feature importances')
save_fig('06_feature_importance.png')
plt.show()
print(importance.head(10).to_string(index=False))

In [ ]:
# 5. Business-readable summary
best_model = min(results, key=lambda k: results[k]['rmspe'])
best_metrics = results[best_model]

summary_lines = [
    "=== BUSINESS SUMMARY ===",
    "Business question: what is the expected demand for each store over the next 8 weeks?",
    "",
    f"Best model: {best_model}",
    f"RMSPE: {best_metrics['rmspe']:.2f}%",
    f"MAPE:  {best_metrics['mape']:.2f}%",
    f"MAE:   {best_metrics['mae']:.0f} units",
    "",
    "Top demand drivers (from LightGBM feature importance):",
]
try:
    summary_lines += importance.head(5).to_string(index=False).splitlines()
except Exception:
    summary_lines.append("N/A")

summary_lines += [
    "",
    "Recommended action:",
    "- Use the LightGBM model for operational 8-week-ahead demand forecasts.",
    "- Pay extra attention to promo days and the 7-day lag (last week's sales).",
    "- For new stores without history, fall back to the seasonal-naive baseline.",
]
print("\n".join(summary_lines))